In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/tupy-e-bert/config.json
/kaggle/input/tupy-e-bert/training_args.bin
/kaggle/input/tupy-e-bert/tokenizer_config.json
/kaggle/input/tupy-e-bert/model.safetensors
/kaggle/input/tupy-e-bert/special_tokens_map.json
/kaggle/input/tupy-e-bert/vocab.txt
/kaggle/input/tupy-e/binary_test.csv
/kaggle/input/tupy-e/binary_train.csv


In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset as HFDataset
import re
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
from transformers import BertTokenizer, BertModel
import os

# Checkpoint function
def print_checkpoint(message):
    print(f"\n=== Checkpoint: {message} ===\n")
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print_checkpoint("Device Configured")

# Load tokenizer and BERT model
tokenizer = BertTokenizer.from_pretrained("/kaggle/input/tupy-e-bert")
bert_model = BertModel.from_pretrained("/kaggle/input/tupy-e-bert")
print_checkpoint("Tokenizer and BERT Model Loaded")

# Load and preprocess data
train_df = pd.read_csv('/kaggle/input/tupy-e/binary_train.csv')
train_df = train_df[['text', 'hate', 'aggressive']].dropna()
test_df = pd.read_csv('/kaggle/input/tupy-e/binary_test.csv')
test_df = test_df[['text', 'hate', 'aggressive']].dropna()

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+|www.\S+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

train_df['text'] = train_df['text'].apply(preprocess_text)
test_df['text'] = test_df['text'].apply(preprocess_text)
print_checkpoint("Data Preprocessing Completed")

# Compute class weights for imbalanced data
class_counts = train_df['hate'].value_counts().to_dict()
total_samples = len(train_df)
class_weights = torch.tensor([total_samples / (2 * class_counts[i]) for i in [0, 1]], dtype=torch.float).to(device)
print(f"Class weights: {class_weights.tolist()}")
print_checkpoint("Class Weights Computed")

# Prepare labels (integer labels)
train_texts = train_df['text'].tolist()
train_labels = train_df['hate'].tolist()
val_texts = test_df['text'].tolist()
val_labels = test_df['hate'].tolist()
test_texts = test_df['text'].tolist()
test_labels = test_df['hate'].tolist()

# Create datasets using Hugging Face Dataset
train_dataset = HFDataset.from_dict({'text': train_texts, 'labels': train_labels})
val_dataset = HFDataset.from_dict({'text': val_texts, 'labels': val_labels})
test_dataset = HFDataset.from_dict({'text': test_texts, 'labels': test_labels})

# Tokenize
def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
print_checkpoint("Dataset Creation and Tokenization Completed")

# Precompute BERT outputs
def precompute_bert_outputs(dataset, bert_model, device, output_dir, batch_size=16):
    bert_model.eval()
    bert_model.to(device)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    output_files = []
    labels_list = []
    os.makedirs(output_dir, exist_ok=True)

    for idx, batch in tqdm(enumerate(dataloader), total=len(dataloader), desc=f"Precomputing BERT outputs for {output_dir}"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels']

        with torch.no_grad():
            outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
            last_hidden_state = outputs[0]

        output_file = os.path.join(output_dir, f'batch_{idx}.pt')
        torch.save(last_hidden_state.cpu(), output_file)
        output_files.append(output_file)
        labels_list.append(labels.cpu())

        print(f"Batch {idx}: last_hidden_state shape: {last_hidden_state.shape}, labels shape: {labels.shape}")

    labels = torch.cat(labels_list, dim=0)
    return output_files, labels

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_output_dir_train = "/kaggle/working/bert_outputs/train"
bert_output_dir_val = "/kaggle/working/bert_outputs/val"
bert_output_dir_test = "/kaggle/working/bert_outputs/test"

train_output_files, train_labels = precompute_bert_outputs(
    train_dataset, bert_model, device, bert_output_dir_train, batch_size=16
)
val_output_files, val_labels = precompute_bert_outputs(
    val_dataset, bert_model, device, bert_output_dir_val, batch_size=16
)
test_output_files, test_labels = precompute_bert_outputs(
    test_dataset, bert_model, device, bert_output_dir_test, batch_size=16
)

print(f"Train labels shape: {train_labels.shape}, Number of BERT output files: {len(train_output_files)}")
print(f"Val labels shape: {val_labels.shape}, Number of BERT output files: {len(val_output_files)}")
print(f"Test labels shape: {test_labels.shape}, Number of BERT output files: {len(test_output_files)}")
print_checkpoint("BERT Outputs Precomputation Completed and Files Created")

# Define custom PyTorch Dataset for precomputed outputs
class BertOutputDataset(Dataset):
    def __init__(self, output_files, labels):
        self.output_files = output_files
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        batch_size = 16
        batch_idx = idx // batch_size
        within_batch_idx = idx % batch_size
        batch_hidden_state = torch.load(self.output_files[batch_idx])
        hidden_state = batch_hidden_state[within_batch_idx]
        label = self.labels[idx]
        return {'hidden_state': hidden_state, 'labels': label}

train_bert_dataset = BertOutputDataset(train_output_files, train_labels)
val_bert_dataset = BertOutputDataset(val_output_files, val_labels)
test_bert_dataset = BertOutputDataset(test_output_files, test_labels)

train_dataloader = DataLoader(train_bert_dataset, batch_size=16, shuffle=True)
val_dataloader = DataLoader(val_bert_dataset, batch_size=16)
test_dataloader = DataLoader(test_bert_dataset, batch_size=16)
print_checkpoint("DataLoaders for Precomputed Outputs Created")

# Define AttentionPooling
class AttentionPooling(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1)

    def forward(self, lstm_output):  # lstm_output: (batch, seq_len, hidden)
        scores = self.attn(lstm_output).squeeze(-1)          # (batch, seq_len)
        weights = torch.softmax(scores, dim=1)               # (batch, seq_len)
        context = torch.sum(lstm_output * weights.unsqueeze(-1), dim=1)  # (batch, hidden)
        return context

# Define enhanced model (BiLSTM + Attention + Classifier)
class BiLSTMAttnModel(nn.Module):
    def __init__(self, hidden_size=768, lstm_hidden_size=256, num_labels=2):
        super(BiLSTMAttnModel, self).__init__()
        self.dropout = nn.Dropout(0.1)
        self.bilstm = nn.LSTM(hidden_size, lstm_hidden_size, num_layers=2, batch_first=True, bidirectional=True, dropout=0.1)
        self.attn_pool = AttentionPooling(lstm_hidden_size * 2)
        self.intermediate = nn.Sequential(
            nn.Linear(lstm_hidden_size * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.1)
        )
        self.classifier = nn.Linear(256, num_labels)

    def forward(self, hidden_state):
        hidden_state = self.dropout(hidden_state)  # (batch, seq_len, hidden)
        lstm_output, _ = self.bilstm(hidden_state)  # (batch, seq_len, 2*lstm_hidden_size)
        context_vector = self.attn_pool(lstm_output)  # (batch, 2*lstm_hidden_size)
        x = self.intermediate(context_vector)  # (batch, 256)
        logits = self.classifier(x)  # (batch, num_labels)
        return logits

# Instantiate model
custom_model = BiLSTMAttnModel(hidden_size=768, lstm_hidden_size=256, num_labels=2)
custom_model.to(device)
print_checkpoint("Model Instantiated")

# Optimizer
optimizer = optim.AdamW([
    {"params": custom_model.bilstm.parameters(), "lr": 1e-3},
    {"params": custom_model.attn_pool.parameters(), "lr": 1e-3},
    {"params": custom_model.intermediate.parameters(), "lr": 1e-3},
    {"params": custom_model.classifier.parameters(), "lr": 1e-3}
])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1, verbose=True)
print_checkpoint("Optimizer and Scheduler Configured")

# Loss function with class weights
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Training loop
num_epochs = 5
custom_model.train()
model_checkpoint_dir = "/kaggle/working/checkpoints"
os.makedirs(model_checkpoint_dir, exist_ok=True)
best_val_accuracy = 0.0
best_model_path = "/kaggle/working/my-trained-bilstm-attn-model/best_model.pt"
patience = 2
early_stop_counter = 0

for epoch in tqdm(range(num_epochs), desc="Training Epochs"):
    total_loss = 0
    train_predictions, train_true_labels = [], []
    for batch in tqdm(train_dataloader, desc=f"Training Epoch {epoch+1}", leave=False):
        hidden_state = batch['hidden_state'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        logits = custom_model(hidden_state)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        train_predictions.extend(preds.cpu().numpy())
        train_true_labels.extend(labels.cpu().numpy())

        print(f"Batch shapes: hidden_state={hidden_state.shape}, logits={logits.shape}, labels={labels.shape}")

    avg_loss = total_loss / len(train_dataloader)
    train_accuracy = accuracy_score(train_true_labels, train_predictions)
    print(f"Epoch {epoch+1}/{num_epochs}, Average Loss: {avg_loss:.4f}, Train Accuracy: {train_accuracy:.4f}")

    # Validation (using test_df)
    custom_model.eval()
    predictions, true_labels = [], []
    with torch.no_grad():
        for batch in tqdm(val_dataloader, desc="Validating", leave=False):
            hidden_state = batch['hidden_state'].to(device)
            labels = batch['labels'].to(device)

            logits = custom_model(hidden_state)
            preds = torch.argmax(logits, dim=1)

            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    val_accuracy = accuracy_score(true_labels, predictions)
    val_f1 = f1_score(true_labels, predictions, average='binary')
    print(f"Validation Accuracy: {val_accuracy:.4f}, F1 Score: {val_f1:.4f}")

    # Learning rate scheduler step
    scheduler.step(val_accuracy)

    # Save best model
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        os.makedirs(os.path.dirname(best_model_path), exist_ok=True)
        torch.save(custom_model.state_dict(), best_model_path)
        print(f"Saved best model with accuracy {val_accuracy:.4f} at {best_model_path}")
        early_stop_counter = 0
    else:
        early_stop_counter += 1

    # Early stopping
    if early_stop_counter >= patience:
        print(f"Early stopping triggered after {epoch+1} epochs")
        break

    custom_model.train()

    # Save epoch checkpoint
    checkpoint_path = os.path.join(model_checkpoint_dir, f"epoch_{epoch+1}_model.pt")
    torch.save(custom_model.state_dict(), checkpoint_path)
    print_checkpoint(f"Epoch {epoch+1} Completed and Model Checkpoint Saved at {checkpoint_path}")

# Load best model for testing
custom_model.load_state_dict(torch.load(best_model_path))
print_checkpoint(f"Best Model Loaded from {best_model_path}")

# Test evaluation (using test_df)
custom_model.eval()
predictions, true_labels = [], []
with torch.no_grad():
    for batch in tqdm(test_dataloader, desc="Testing", leave=False):
        hidden_state = batch['hidden_state'].to(device)
        labels = batch['labels'].to(device)

        logits = custom_model(hidden_state)
        preds = torch.argmax(logits, dim=1)

        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

        print(f"Test batch shapes: hidden_state={hidden_state.shape}, logits={logits.shape}, labels={labels.shape}, preds={preds.shape}")

test_accuracy = accuracy_score(true_labels, predictions)
test_f1 = f1_score(true_labels, predictions, average='binary')
print(f"Test Accuracy: {test_accuracy:.4f}, Test F1 Score: {test_f1:.4f}")
print_checkpoint("Test Evaluation Completed")


=== Checkpoint: Device Configured ===


=== Checkpoint: Tokenizer and BERT Model Loaded ===


=== Checkpoint: Data Preprocessing Completed ===

Class weights: [0.568365216255188, 4.156830310821533]

=== Checkpoint: Class Weights Computed ===



Map:   0%|          | 0/34934 [00:00<?, ? examples/s]

Map:   0%|          | 0/8734 [00:00<?, ? examples/s]

Map:   0%|          | 0/8734 [00:00<?, ? examples/s]


=== Checkpoint: Dataset Creation and Tokenization Completed ===



...


Training Epoch 5:  15%|█▌        | 332/2184 [00:17<01:37, 19.07it/s]

Batch shapes: hidden_state=torch.Size([16, 128, 768]), logits=torch.Size([16, 2]), labels=torch.Size([16])
Batch shapes: hidden_state=torch.Size([16, 128, 768]), logits=torch.Size([16, 2]), labels=torch.Size([16])



Training Epoch 5:  15%|█▌        | 334/2184 [00:18<01:36, 19.12it/s]

Batch shapes: hidden_state=torch.Size([16, 128, 768]), logits=torch.Size([16, 2]), labels=torch.Size([16])
Batch shapes: hidden_state=torch.Size([16, 128, 768]), logits=torch.Size([16, 2]), labels=torch.Size([16])



Training Epoch 5:  15%|█▌        | 336/2184 [00:18<01:36, 19.15it/s]